# Chapter 1 · Data Exploration & Cleaning

```{div} chapter-meta
<span>📊 Part I: Working with Data</span>
<span>⏱ 3–4 hours</span>
<span>🔧 pandas · numpy · matplotlib · seaborn</span>
```

Before building any model, you must *understand* your data.
Raw datasets are almost always flawed: values are missing, columns are encoded inconsistently, outliers lurk, and some entries are simply impossible.
In this chapter you will learn how to **explore** data systematically using statistics and visualizations, then **clean** it so that downstream analyses are reliable.

We work throughout with a real-world-style HR dataset (`flawed_dataset.csv`) that ships with intentional defects — perfect for practicing every technique covered here.


```{raw} html
<div style="text-align:right; margin-bottom:0.5rem">
  <a href="../../_static/slides/ch1.html"
     target="_blank"
     style="display:inline-block; background:#2962ff; color:#fff;
            border-radius:6px; padding:0.35em 1em; text-decoration:none;
            font-size:0.85rem; font-weight:600">
    &#9654; View as Slideshow
  </a>
</div>
```


::::{grid} 1 1 2 2
:gutter: 3

:::{grid-item-card}
:class-header: sd-bg-primary sd-text-white

**📖 What You'll Learn**
^^^
- Compute and interpret descriptive statistics (mean, median, IQR)
- Choose the right visualization for each data type
- Detect 5 classes of data quality problems
- Build a step-by-step data cleaning pipeline
:::

:::{grid-item-card}
:class-header: sd-bg-secondary sd-text-white

**🛠 Libraries We Use**
^^^
- `pandas` — tabular data manipulation
- `numpy` — numerical computing
- `matplotlib` — foundational plotting
- `seaborn` — statistical visualizations
- `scipy.stats` — statistical functions
:::
::::


## Setup

Run this cell first — it imports all libraries used throughout the chapter and configures a consistent visual style.


In [ ]:
import statistics
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (8, 4)})
pd.set_option('display.max_columns', 12)
pd.set_option('display.float_format', '{:.2f}'.format)

---

## Meet the Dataset

Our dataset describes 1 050 (fictional) employees. It has been deliberately corrupted in ways that mirror common real-world problems.

::::{grid} 2 3 5 5
:gutter: 2

:::{grid-item-card} ID
:class-header: sd-bg-dark sd-text-white
Employee identifier
:::

:::{grid-item-card} Name
:class-header: sd-bg-dark sd-text-white
First name (sometimes blank)
:::

:::{grid-item-card} Age
:class-header: sd-bg-dark sd-text-white
Age in years — contains `-5` and `999` sentinels
:::

:::{grid-item-card} Gender
:class-header: sd-bg-dark sd-text-white
Mixed case: `f`, `F`, `female`, `FEMALE`
:::

:::{grid-item-card} Email
:class-header: sd-bg-dark sd-text-white
Many rows say `"invalid-email"`
:::

:::{grid-item-card} Join_Date
:class-header: sd-bg-dark sd-text-white
Some rows contain `"not-a-date"`
:::

:::{grid-item-card} Salary
:class-header: sd-bg-dark sd-text-white
Sentinels: `-1000` and `1 000 000`
:::

:::{grid-item-card} Department
:class-header: sd-bg-dark sd-text-white
`HR`, `hr`, `Human Resources` all mixed
:::

:::{grid-item-card} Feedback_Score
:class-header: sd-bg-dark sd-text-white
0–10 scale; extremes may be placeholders
:::

:::{grid-item-card} Left_Company
:class-header: sd-bg-dark sd-text-white
`0/1`, `Yes/No`, `Y/N`, `y/n` all mixed
:::
::::


In [ ]:
df = pd.read_csv('flawed_dataset.csv')
print(f'Shape: {df.shape}  ({df.shape[0]} rows × {df.shape[1]} columns)')
df.head(10)

In [ ]:
df.info()

In [ ]:
missing = df.isnull().sum()
pct     = (missing / len(df) * 100).round(1)
(
    pd.DataFrame({'Missing Count': missing, 'Missing %': pct})
    .sort_values('Missing Count', ascending=False)
)

---

## Section 1 — Descriptive Statistics

*Descriptive* statistics summarize data in single numbers so we can understand its key properties at a glance.
They are the first tool in every data scientist's toolkit.

::::{grid} 1 2 3 3
:gutter: 3

:::{grid-item-card} Central Tendency
:class-header: sd-bg-primary sd-text-white

**Mean · Median · Mode**
^^^
Where is the *center* of the data?

- **Mean** — sum ÷ count; sensitive to outliers
- **Median** — middle value when sorted; robust
- **Mode** — most frequent value; for categorical data
:::

:::{grid-item-card} Variability
:class-header: sd-bg-success sd-text-white

**Std Dev · IQR**
^^^
How *spread out* is the data?

- **Std Dev** — average distance from the mean; parametric
- **IQR** — range of the middle 50 %; robust to outliers
:::

:::{grid-item-card} Range
:class-header: sd-bg-warning sd-text-white

**Min · Max**
^^^
What are the *boundary* values?

Deceptively powerful: a minimum of `-5` for Age immediately flags impossible entries without any model.
:::
::::


### Central Tendency

The **arithmetic mean** is defined as

$$\text{mean}(x) = \frac{1}{n}\sum_{i=1}^{n} x_i$$

The **median** is the middle value in sorted data — 50 % of values lie below it and 50 % above.
Unlike the mean it makes no assumptions about the distribution, making it *robust to outliers*.

The **mode** is the most frequent value and works for categorical data where mean and median are undefined.


In [ ]:
# Use Age after filtering out sentinel values for this demonstration
age_valid = df['Age'][(df['Age'] >= 0) & (df['Age'] <= 100)].dropna()

print('=== Central Tendency of Age (valid values only) ===')
print(f'  Mean  : {age_valid.mean():.1f} years')
print(f'  Median: {age_valid.median():.1f} years')
print(f'  Mode  : {age_valid.mode().values[0]:.0f} years')
print()

# Department is categorical — only mode makes sense
dept_valid = df['Department'].dropna()
print('=== Central Tendency of Department (categorical) ===')
print(f'  Mode  : {statistics.mode(dept_valid)}')

In [ ]:
# Classic demonstration: mean collapses under one extreme outlier; median barely moves
data             = [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]
data_with_outlier = data + [100]

print('Without outlier:')
print(f'  mean   = {statistics.mean(data):.2f}')
print(f'  median = {statistics.median(data):.2f}')

print('\nWith a single outlier (value = 100):')
print(f'  mean   = {statistics.mean(data_with_outlier):.2f}  ← shifted dramatically')
print(f'  median = {statistics.median(data_with_outlier):.2f} ← barely changed')

### Variability

The **standard deviation** measures average spread around the mean:

$$\text{sd}(x) = \sqrt{\frac{\sum_{i=1}^{n}(x_i - \bar{x})^2}{n-1}}$$

For normally distributed data, 68 % of values lie within ±1 sd, 95 % within ±2 sd, and 99.7 % within ±3 sd (the *68-95-99.7 rule*).

The **interquartile range (IQR)** is the distance between the 75th and 25th percentiles:

$$\text{IQR}(x) = Q_{0.75} - Q_{0.25}$$

Because IQR only considers the middle 50 % of the data it is robust to extreme values — the non-parametric counterpart to the standard deviation.


In [ ]:
data             = [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]
data_with_outlier = data + [100]

print('Without outlier:')
print(f'  std dev = {statistics.stdev(data):.2f}')
print(f'  IQR     = {scipy_stats.iqr(data):.2f}')

print('\nWith outlier (value = 100):')
print(f'  std dev = {statistics.stdev(data_with_outlier):.2f}  ← exploded')
print(f'  IQR     = {scipy_stats.iqr(data_with_outlier):.2f} ← stable')

### Range — Your First Data Quality Check

Min and max values are trivial to compute but surprisingly revealing.
Impossible values that would slip past central-tendency checks are immediately visible in the range.


In [ ]:
for col in ['Age', 'Salary', 'Feedback_Score']:
    print(f'{col}: min = {df[col].min()},  max = {df[col].max()}')

print()
print('Findings:')
print('  Age     min=-5    → impossible (sentinel for missing)')
print('  Age     max=999   → impossible (sentinel for missing)')
print('  Salary  min=-1000 → impossible (placeholder value)')
print('  Salary  max=1e6   → implausible (placeholder value)')

In [ ]:
# pandas .describe() shows all of the above at once for numeric columns
df.describe().round(1)

---

## Section 2 — Visualization

Numbers tell part of the story. Visualizations reveal the *shape* of data — patterns, clusters, and anomalies that descriptive statistics miss.

::::{grid} 1 2 2 4
:gutter: 2

:::{grid-item-card} Histogram
:class-header: sd-bg-primary sd-text-white

**Distribution of one feature**
^^^
Counts values in equal-width bins. Combine with a KDE curve for a smooth estimate of the density.
:::

:::{grid-item-card} Box Plot
:class-header: sd-bg-success sd-text-white

**5-number summary**
^^^
Shows median, Q1, Q3, whiskers (±1.5 × IQR), and individual outlier dots.
:::

:::{grid-item-card} Pair Plot
:class-header: sd-bg-warning sd-text-white

**All pairwise relationships**
^^^
A matrix of scatter plots (off-diagonal) and histograms (diagonal). Ideal for spotting correlations.
:::

:::{grid-item-card} Correlation Heatmap
:class-header: sd-bg-danger sd-text-white

**Numeric correlations at a glance**
^^^
Color-encodes correlation coefficients (–1 to +1). Scales to many features where pair plots become cluttered.
:::
::::


### Why Visualization Matters — Anscombe's Quartet

All four datasets below have the same mean, standard deviation, and linear regression line ($y = 3 + 0.5x$).
Yet they are completely different in structure — a fact invisible from statistics alone.

```{admonition} Key Insight
:class: tip
Always visualize your data. Statistics can be identical for datasets with radically different shapes.
```


In [ ]:
x  = np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5])
y1 = np.array([8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68])
y2 = np.array([9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74])
y3 = np.array([7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73])
x4 = np.array([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8])
y4 = np.array([6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89])

xfit = np.array([4, 20])
yfit = 3 + 0.5 * xfit

fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharey=True, sharex=True)
pairs = [(x, y1, '$x_1, y_1$  — roughly linear'),
         (x, y2, '$x_2, y_2$  — curved relationship'),
         (x, y3, '$x_3, y_3$  — one outlier distorts the line'),
         (x4, y4, '$x_4, y_4$  — x is nearly constant')]

for ax, (xi, yi, label) in zip(axes.flat, pairs):
    ax.plot(xi, yi, color='darkorange', marker='o', linestyle='none', ms=7, alpha=0.9)
    ax.plot(xfit, yfit, color='steelblue', lw=1.5, label='y = 3 + 0.5x')
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('x'); ax.set_ylabel('y')

fig.suptitle("Anscombe's Quartet — identical statistics, completely different data",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### Visualizing Distributions — Age & Salary

Histograms combined with a KDE curve and a rug plot give the most complete picture of a single numeric feature.
Notice how the sentinel values (`-5`, `999`, `-1000`, `1 000 000`) spike conspicuously at the boundaries.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: raw Age including sentinel values
axes[0].hist(df['Age'].dropna(), bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Age (raw) — spikes at -5 and 999 reveal sentinels')
axes[0].set_xlabel('Age');  axes[0].set_ylabel('Count')

# Right: valid range only, with KDE
valid_age = df['Age'][(df['Age'] >= 0) & (df['Age'] <= 100)].dropna()
sns.histplot(valid_age, bins=30, kde=True, ax=axes[1], color='steelblue')
axes[1].set_title('Age (valid range 0–100) — roughly uniform')
axes[1].set_xlabel('Age')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: raw Salary
sns.boxplot(x=df['Salary'].dropna(), ax=axes[0], color='salmon')
axes[0].set_title('Salary (raw) — sentinel outliers dominate the axis')
axes[0].set_xlabel('Salary ($)')

# Right: realistic range only
valid_sal = df['Salary'][(df['Salary'] > 0) & (df['Salary'] < 200_000)].dropna()
sns.histplot(valid_sal, bins=30, kde=True, ax=axes[1], color='salmon')
axes[1].set_title('Salary (filtered $0–$200k) — approximately normal')
axes[1].set_xlabel('Salary ($)')

plt.tight_layout()
plt.show()

In [ ]:
# Build a clean numeric frame to compute correlations
df_num = pd.DataFrame({
    'Age':            df['Age'].where(df['Age'].between(0, 100)),
    'Salary':         df['Salary'].where(df['Salary'].between(0, 200_000)),
    'Feedback_Score': df['Feedback_Score'].where(df['Feedback_Score'].between(0, 10)),
})

fig, ax = plt.subplots(figsize=(6, 5))
cmap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(df_num.corr(), annot=True, fmt='.2f', square=True,
            linewidths=0.5, vmin=-1, vmax=1, cmap=cmap, ax=ax)
ax.set_title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.show()

---

## Section 3 — Diagnosing Data Quality

Before cleaning, we must *find* every problem. Our dataset contains all five canonical data quality issues:

::::{grid} 1 2 3 3
:gutter: 3

:::{grid-item-card} ❓ Missing Values
:class-header: sd-bg-danger sd-text-white

Up to 523 missing per column
^^^
Values left blank or stored as `NaN`. Must be imputed or dropped before modeling.
:::

:::{grid-item-card} 👯 Duplicates
:class-header: sd-bg-warning sd-text-white

50 exact duplicate rows
^^^
Records that appear more than once inflate counts and skew statistics.
:::

:::{grid-item-card} 🔀 Inconsistent Encoding
:class-header: sd-bg-info sd-text-white

`Gender`, `Department`, `Left_Company`
^^^
The same concept written in different ways: `"M"`, `"male"`, `"MALE"`, `"m"` are all the same.
:::

:::{grid-item-card} 📈 Outliers
:class-header: sd-bg-warning sd-text-white

Detected via IQR rule
^^^
Values that are extreme relative to the distribution. May be errors or legitimate rare events.
:::

:::{grid-item-card} 🚫 Sentinel / Invalid Values
:class-header: sd-bg-danger sd-text-white

`-5`, `999`, `-1000`, `1000000`
^^^
Placeholder codes that disguise missing data as numeric values. Must be converted to `NaN`.
:::
::::


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Missingness heatmap
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', ax=axes[0])
axes[0].set_title('Missing Data Heatmap\n(yellow = present, purple = missing)')
axes[0].set_xlabel('Column')
axes[0].set_ylabel('Row index')

# Bar chart of missing counts
missing = df.isnull().sum().sort_values(ascending=True)
missing[missing > 0].plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Missing Value Count per Column')
axes[1].set_xlabel('Missing values')

plt.tight_layout()
plt.show()

In [ ]:
n_dupes = df.duplicated().sum()
print(f'Exact duplicate rows: {n_dupes}')

# Show a sample pair
example = df[df.duplicated(keep=False)].sort_values('ID').head(4)
print('\nExample duplicated records:')
print(example.to_string(index=False))

In [ ]:
print('Gender unique values (before cleaning):')
print(' ', df['Gender'].dropna().unique())

print('\nDepartment unique values (before cleaning):')
print(' ', df['Department'].dropna().unique())

print('\nLeft_Company unique values (before cleaning):')
print(' ', df['Left_Company'].dropna().unique())

In [ ]:
# Sentinel detection — values that are structurally valid but semantically impossible
n_age_sentinel = df['Age'].isin([-5, 999]).sum()
n_sal_sentinel = df['Salary'].isin([-1000, 1_000_000]).sum()

print(f'Age rows with sentinel (-5 or 999)        : {n_age_sentinel}')
print(f'Salary rows with sentinel (-1000 or 1e6)  : {n_sal_sentinel}')

print('\nFeedback_Score distribution at boundaries:')
vc = df['Feedback_Score'].value_counts().sort_index()
print(vc.head(3).to_string())
print('  ...')
print(vc.tail(3).to_string())

In [ ]:
valid_pattern = re.compile(r'^[\w\.\-]+@[\w\.\-]+\.\w+$')

email_status = df['Email'].dropna().apply(
    lambda e: 'valid' if valid_pattern.match(str(e)) else 'invalid'
)

print('Email format breakdown:')
print(email_status.value_counts().to_string())

---

## Section 4 — Data Cleaning Pipeline

Now we fix every problem found above. The pipeline has six steps, applied in order.

::::{grid} 2 3 3 6
:gutter: 2

:::{grid-item-card}
:class-header: sd-bg-dark sd-text-white

**Step 1**
^^^
Remove Duplicates
:::

:::{grid-item-card}
:class-header: sd-bg-dark sd-text-white

**Step 2**
^^^
Standardize Categories
:::

:::{grid-item-card}
:class-header: sd-bg-dark sd-text-white

**Step 3**
^^^
Sentinel → NaN
:::

:::{grid-item-card}
:class-header: sd-bg-dark sd-text-white

**Step 4**
^^^
IQR Outlier Filter
:::

:::{grid-item-card}
:class-header: sd-bg-dark sd-text-white

**Step 5**
^^^
Parse & Validate
:::

:::{grid-item-card}
:class-header: sd-bg-dark sd-text-white

**Step 6**
^^^
Impute Missing
:::
::::


In [ ]:
# ── Step 1 · Remove exact duplicate rows ──────────────────────────────────────
df_clean = df.copy()

before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after  = len(df_clean)

print(f'Removed {before - after} duplicate rows  ({before} → {after})')

In [ ]:
# ── Step 2 · Standardize categorical columns ───────────────────────────────────

# Gender: collapse all variants to 'female' / 'male'
df_clean['Gender'] = (
    df_clean['Gender'].str.lower().str.strip()
    .replace({'f': 'female', 'm': 'male'})
)
print('Gender:', sorted(df_clean['Gender'].dropna().unique()))

# Department: title-case then unify abbreviations
dept_map = {
    'Hr': 'Human Resources', 'HR': 'Human Resources',
    'Eng': 'Engineering',    'ENG': 'Engineering',    'ENGINEERING': 'Engineering',
    'Fin': 'Finance',        'FIN': 'Finance',        'FINANCE': 'Finance',
}
df_clean['Department'] = (
    df_clean['Department'].str.strip().str.title()
    .replace(dept_map)
)
print('Department:', sorted(df_clean['Department'].dropna().unique()))

# Left_Company: collapse yes/no variants to 1 / 0
lc_map = {'yes': 1, 'y': 1, '1': 1, '1.0': 1, 'no': 0, 'n': 0, '0': 0, '0.0': 0, 'nan': np.nan}
df_clean['Left_Company'] = (
    df_clean['Left_Company'].astype(str).str.lower().str.strip()
    .replace(lc_map)
)
df_clean['Left_Company'] = pd.to_numeric(df_clean['Left_Company'], errors='coerce')
print('Left_Company:', sorted(df_clean['Left_Company'].dropna().unique()))

In [ ]:
# ── Step 3 · Replace sentinel / impossible values with NaN ────────────────────

# Age: -5 and 999 are not real ages
df_clean['Age'] = df_clean['Age'].where(df_clean['Age'].between(0, 90))

# Salary: -1000 and 1 000 000 are placeholder codes
df_clean['Salary'] = df_clean['Salary'].where(df_clean['Salary'].between(10_000, 200_000))

print('Missing after sentinel removal:')
print(df_clean[['Age', 'Salary']].isnull().sum())

In [ ]:
# ── Step 4 · IQR outlier filter ────────────────────────────────────────────────

def iqr_filter(series: pd.Series) -> pd.Series:
    """Set values outside Q1 - 1.5*IQR … Q3 + 1.5*IQR to NaN."""
    Q1, Q3 = series.quantile([0.25, 0.75])
    IQR    = Q3 - Q1
    return series.where(series.between(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR))

for col in ['Age', 'Salary']:
    before = df_clean[col].notna().sum()
    df_clean[col] = iqr_filter(df_clean[col])
    after  = df_clean[col].notna().sum()
    print(f'{col}: IQR filter removed {before - after} additional outliers')

In [ ]:
# ── Step 5 · Parse dates & validate email formats ─────────────────────────────

# Dates: 'not-a-date' and blank become NaT
df_clean['Join_Date'] = pd.to_datetime(df_clean['Join_Date'], errors='coerce')
print(f'Join_Date — invalid/missing: {df_clean["Join_Date"].isnull().sum()}')

# Emails: mark 'invalid-email' and malformed addresses as NaN
valid_email_re = re.compile(r'^[\w\.\-]+@[\w\.\-]+\.\w+$')
df_clean['Email'] = df_clean['Email'].where(
    df_clean['Email'].str.match(valid_email_re, na=False)
)
print(f'Email     — invalid/missing: {df_clean["Email"].isnull().sum()}')

In [ ]:
# ── Step 6 · Impute missing values ─────────────────────────────────────────────

# Numeric: median imputation (robust to remaining skew)
df_clean['Age']            = df_clean['Age'].fillna(df_clean['Age'].median())
df_clean['Feedback_Score'] = df_clean['Feedback_Score'].fillna(df_clean['Feedback_Score'].median())

# Salary: department-wise median (captures pay-grade differences)
df_clean['Salary'] = df_clean.groupby('Department')['Salary'].transform(
    lambda s: s.fillna(s.median())
)
df_clean['Salary'] = df_clean['Salary'].fillna(df_clean['Salary'].median())  # fallback

# Categorical: fill with a placeholder so models don't see NaN
df_clean['Name']       = df_clean['Name'].fillna('Unknown')
df_clean['Department'] = df_clean['Department'].fillna('Unknown')
df_clean['Gender']     = df_clean['Gender'].fillna('unknown')

# Binary: mode imputation
df_clean['Left_Company'] = df_clean['Left_Company'].fillna(df_clean['Left_Company'].mode()[0])

# Join_Date: fill with a sentinel date so downstream code doesn't need null-handling
df_clean['Join_Date'] = df_clean['Join_Date'].fillna(pd.Timestamp('2000-01-01'))

# Email column: too sparse (< 30 % valid) — drop it
df_clean = df_clean.drop(columns=['Email'])

print('Missing values after imputation:')
print(df_clean.isnull().sum())

In [ ]:
# ── Before / After Summary ─────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Before': [
        df.shape[0],
        int(df.duplicated().sum()),
        int(df.isnull().sum().sum()),
        f"{df['Age'].min():.0f} – {df['Age'].max():.0f}",
        f"${df['Salary'].min():.0f} – ${df['Salary'].max():.0f}",
        ', '.join(sorted(df['Gender'].dropna().unique())),
        df['Department'].dropna().nunique(),
    ],
    'After': [
        df_clean.shape[0],
        int(df_clean.duplicated().sum()),
        int(df_clean.isnull().sum().sum()),
        f"{df_clean['Age'].min():.0f} – {df_clean['Age'].max():.0f}",
        f"${df_clean['Salary'].min():.0f} – ${df_clean['Salary'].max():.0f}",
        ', '.join(sorted(df_clean['Gender'].unique())),
        df_clean['Department'].nunique(),
    ]
}, index=[
    'Row count', 'Duplicates', 'Total NaN cells',
    'Age range', 'Salary range',
    'Gender values', 'Department values'
])

summary

In [ ]:
# Visualize the cleaned distributions side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(df_clean['Age'],            bins=25, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Age (cleaned)')

sns.histplot(df_clean['Salary'],         bins=25, kde=True, ax=axes[1], color='salmon')
axes[1].set_title('Salary (cleaned)')

sns.countplot(data=df_clean, x='Gender', ax=axes[2],
              order=['female', 'male', 'unknown'], palette='muted')
axes[2].set_title('Gender (cleaned)')

plt.tight_layout()
plt.show()

---

## Exercises

```{admonition} Exercise 1 — Spot the Skew
:class: tip
Plot the histogram and box plot of `Feedback_Score` **before** cleaning.
Do the extreme values (0 and 10) look like real scores or sentinel codes?
Justify your answer using the descriptive statistics.
```

```{admonition} Exercise 2 — Robustness Check
:class: tip
Compute the mean and median of `Salary` *before* removing sentinel values.
Then compute them again *after* Step 3.
Which statistic was more misleading before cleaning, and by how much?
```

```{admonition} Exercise 3 — Department Imputation Strategy
:class: tip
In Step 6 we imputed `Salary` using the **department-wise median**.
Why is this better than the global median?
Demonstrate by computing both and comparing the imputed salary for a row
in `'Engineering'` versus `'Finance'`.
```

````{admonition} Exercise 4 — Anscombe's Own Data
:class: tip
Compute the mean, standard deviation, and Pearson correlation for all four
Anscombe datasets ($x_1,y_1$ through $x_4,y_4$).
Confirm they are (nearly) identical, then explain in one paragraph why
this makes visualization indispensable.
````


---

## Summary

::::{grid} 1 2 2 2
:gutter: 3

:::{grid-item-card} What We Covered
:class-header: sd-bg-primary sd-text-white

^^^
| Concept | Key Rule |
|---|---|
| Mean vs Median | Use median when outliers are present |
| Std Dev vs IQR | Use IQR when using the median |
| Range | Always check — catches impossible values instantly |
| Visualization | Always plot — statistics can mislead (Anscombe) |
| Missing Values | Heatmap first, then impute by column type |
| Duplicates | `drop_duplicates()` before any analysis |
| Sentinels | Convert to `NaN`, then treat as missing |
| Outliers | IQR rule: flag values beyond Q1/Q3 ± 1.5 × IQR |
:::

:::{grid-item-card} Cleaning Order Matters
:class-header: sd-bg-success sd-text-white

^^^
Always clean in this sequence:

1. **Deduplicate** — so imputation isn't skewed by repeated rows
2. **Standardize categories** — so group-wise statistics are accurate
3. **Remove sentinels** — before computing IQR bounds
4. **Filter outliers** — after sentinels are gone
5. **Parse dates / validate formats** — lightweight transformations
6. **Impute** — last, so you use clean values as the basis
:::
::::

```{admonition} What's Next
:class: note
**Chapter 2 — NumPy** dives into the array computing layer that powers pandas and every machine learning library.
You will use the cleaned dataset from this chapter in later chapters on regression, classification, and feature engineering.
```
